In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine,types as sqltypes
import urllib
import pyodbc

In [3]:
#تحميل الداتا
fact_assessments = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx")
dim_students = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_students')
dim_teachers = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_teachers')
dim_courses = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_courses')
dim_classes = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_classes')
dim_departments = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_departments')
dim_campuses = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_campuses')
dim_terms = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_terms')
dim_guardians = pd.read_excel(R"C:\Users\K.Asus\Desktop\programing\Data_Analysis\TEST\المدرسه\school_dataset.xlsx", sheet_name='dim_guardians')


In [4]:
fact_assessments .info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13485 entries, 0 to 13484
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   record_id        13485 non-null  int64         
 1   learner_id       13485 non-null  int64         
 2   section_code     13485 non-null  object        
 3   exam_dt          13485 non-null  datetime64[ns]
 4   assessment_kind  13485 non-null  object        
 5   score_points     13067 non-null  float64       
 6   max_mark         13485 non-null  int64         
 7   presence_flag    13485 non-null  object        
 8   grade_txt        13097 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 948.3+ KB


In [5]:
# توحيد الماكس مارك لـ 100
fact_assessments['max_mark'] = 100

In [6]:
#تقيميات (امتياز - جيد جدا - جيد - مقبول )
def calculate_grade(score):
    if score >= 90: return 'A+'
    elif score >= 85: return 'A'
    elif score >= 80: return 'B+'
    elif score >= 75: return 'B'
    elif score >= 70: return 'C+'
    elif score >= 65: return 'C'
    elif score >= 60: return 'D'
    else: return 'F'

In [7]:
#لو الايميل فاضي خد اسم المدرس + الكود بتاعه (ايميل افتراضي)
def fix_teacher_email(row):
    if pd.isna(row['teacher_email']) or str(row['teacher_email']).strip() == '':
        clean_name = str(row['teacher_name']).replace(" ", "").lower()
        new_email = clean_name + str(row['teacher_no']) + "@gmail.com"
        return new_email
        return row['teacher_email']
dim_teachers['teacher_email'] = dim_teachers.apply(fix_teacher_email, axis=1)

In [8]:
#لو الرقم ناقص زود 0 في الاول 
def clean_phone(phone):
    phone = str(phone).strip()
    if len(phone) == 10 and phone.startswith('1'):
        return '0' + phone
    return phone
dim_students['mobile_num'] = dim_students['mobile_num'].apply(clean_phone)

In [9]:
# تنضيف أسماء الأعمدة نفسها
for df in [fact_assessments, dim_students, dim_teachers]:
    df.columns = df.columns.str.strip() 
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()

In [10]:
##ليست بكل الجداول 
all_dfs = [fact_assessments, dim_students, dim_teachers, dim_courses, 
           dim_classes, dim_departments, dim_campuses, dim_terms, dim_guardians]

In [11]:
#مسح اي مسافات في اسماء الاعمده 
for df in all_dfs:
    df.columns = df.columns.str.strip()

In [12]:
#مسح اي مسافات زياده داخل الاعمد 
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
  df[col] = df[col].astype(str).str.strip()

In [13]:
#دي بقا توحيد اسم النوع 
if 'gender_code' in dim_students.columns:
    gender_map = {'M': 'Male', 'male': 'Male', 'F': 'Female', 'female': 'Female'}
    dim_students['gender_code'] = dim_students['gender_code'].map(gender_map).fillna('Other')

In [14]:
tables_to_test = {
    "Fact Assessments": fact_assessments,
    "Students": dim_students,
    "Teachers": dim_teachers,
    "Courses": dim_courses,
    "Classes": dim_classes,
    "Departments": dim_departments,
    "Campuses": dim_campuses,
    "Terms": dim_terms,
    "Guardians": dim_guardians
}
for name, df in tables_to_test.items():
    print("Table: " + name)
    col_spaces = 0
    for c in df.columns:
        if c != c.strip():
            col_spaces = col_spaces + 1          
    data_spaces = 0
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        data_spaces = data_spaces + df[col].apply(lambda x: 1 if str(x).startswith(' ') or str(x).endswith(' ') else 0).sum()
    print("Column Spaces: " + str(col_spaces))
    print("Data Spaces: " + str(data_spaces))
    if name == "Fact Assessments":
        wrong_marks = df[df['max_mark'] != 100].shape[0]
        print("Max Mark Errors: " + str(wrong_marks))   
    print("-------------------")
print("ميه ميه الدلتا  دي صح الصح ")

Table: Fact Assessments
Column Spaces: 0
Data Spaces: 0
Max Mark Errors: 0
-------------------
Table: Students
Column Spaces: 0
Data Spaces: 0
-------------------
Table: Teachers
Column Spaces: 0
Data Spaces: 0
-------------------
Table: Courses
Column Spaces: 0
Data Spaces: 6
-------------------
Table: Classes
Column Spaces: 0
Data Spaces: 12
-------------------
Table: Departments
Column Spaces: 0
Data Spaces: 0
-------------------
Table: Campuses
Column Spaces: 0
Data Spaces: 0
-------------------
Table: Terms
Column Spaces: 0
Data Spaces: 0
-------------------
Table: Guardians
Column Spaces: 0
Data Spaces: 0
-------------------
ميه ميه الدلتا  دي صح الصح 


In [15]:
#رفع الداتا علي SQL (باستخدام جيمناي )
server = 'DESKTOP-VAH6K54\\sqlexpress'
database = 'TEST'
connection_string = f"mssql+pyodbc://@{server}/{database}?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server"
engine = sqlalchemy.create_engine(connection_string)
print("تم الاتصال بـ SQL Server")
tables = {
    "fact_assessments": fact_assessments,
    "dim_students": dim_students,
    "dim_teachers": dim_teachers,
    "dim_courses": dim_courses,
    "dim_classes": dim_classes,
    "dim_departments": dim_departments,
    "dim_campuses": dim_campuses,
    "dim_terms": dim_terms,
    "dim_guardians": dim_guardians
}
for table_name, df in tables.items():
    try:
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='replace',
            index=False,
            chunksize=1000
        )
        print(f" تم رفع جدول {table_name}")
    except Exception as e:
        print(f" حصل مشكلة في جدول {table_name}")
        print(e)

print(" كل الداتا اترفعت بنجاح")

NameError: name 'sqlalchemy' is not defined

In [ ]:
#معرفه عدد واسماء الجداول اللي موجوده في القاعده 
import pandas as pd

query = "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES"
tables_in_db = pd.read_sql(query, engine)

print(tables_in_db)